# Building models

Models are records of `Nn` layers. A `Seq` chain composes them with `(~~>)`, threading
dimensions through the type — each layer's output dim must equal the next's input dim,
checked at compile time. The `Seq` type pins only its endpoints; hidden dims are
existential.

Two types from tutorial 01:
- `Array [dims] ty` — pure-Idris structural data (Vect-of-Vect). What you build host data with.
- `Tensor dims ex dt g` — the autograd handle on a backend. What weights and forward outputs are.

## Layer constructors

Each layer with parameters is built in the `Init` monad (it allocates + registers its
params). `linear {i} {o}` returns `Init (Linear i o ex dt g)`. Stateless activations
(`reluA`, `tanhA`, `sigmoidA`, `geluA`) are plain values.

In [ ]:
:t linear

In [ ]:
:t reluA

## Composition with `(~~>)`

Build a one-hidden-layer MLP inside `Init`, then realise it with `runInitL` (which
populates the C param registry, naming params from the scope path — no manual prefixes).
We use a `:let` definition so the model is reusable across cells:

In [ ]:
mkMlp : Init (Seq 2 3 TapeExecutor F64 WithGrad)
mkMlp = do { l1 <- linear {i=2} {o=8}; l2 <- linear {i=8} {o=3}; pure (l1 ~~> reluA ~~> l2 ~~> Nil) }

In [ ]:
:exec run (do { m <- runInitL mkMlp; discard m; liftIO1 (putStrLn "MLP built (2 -> 8 -> 3).") })

## Shape mismatches are compile errors

If one layer's output dim doesn't match the next's input, the chain won't elaborate — no
runtime crash, no silent broadcast. The next cell is *expected to fail*: `l1` outputs 8 but
`l2` expects 5, so the composition can't be typed (the compiler can't even resolve the
chain's hidden dimension):

In [ ]:
:exec run (do {
  m <- runInitL (the (Init (Seq 4 3 TapeExecutor F64 WithGrad))
                     (do { l1 <- linear {i=4} {o=8}; l2 <- linear {i=5} {o=3};
                           pure (l1 ~~> l2 ~~> Nil) }));
  discard m; liftIO1 (putStrLn "should not reach here") })

## Forward pass

`forwardSeq` threads a batched `Tensor [b, i]` input through the chain to a
`Tensor [b, o]` output. The model is a **linear resource**: `forwardSeq` consumes it and
returns it (the output rides a `(!*)` bang), so you thread the handle through:

In [ ]:
:t forwardSeq

In [ ]:
:exec run (do {
  m <- runInitL mkMlp;
  x <- liftIO1 (tensor {dims=[1,2]} {ex=TapeExecutor} {dt=F64} (FromVect [1.0, 2.0]));
  (MkBang out # m1) <- forwardSeq {b=1} m (retypeGrad x);
  discard m1;
  liftIO1 (putStrLn ("output[0,0] = " ++ show (primItem2d {ex=TapeExecutor} out.tensorPtr 0 0))) })

## Available layers

Every parameterised layer is an `Init` builder; stateless activations are plain values:

In [ ]:
:t lstm

In [ ]:
:t gru

In [ ]:
:t conv2d

In [ ]:
:t dropout

In [ ]:
:t embedding

In [ ]:
:t tanhA

## Multi-network parameter scoping

For multi-network setups (A2C / PPO / SAC actor + critic, DQN online + target), build each
network as its own `Init` sub-model and use `Nn.Group.groupOf submodel` to recover that
submodel's exact registry names for per-network optimizer scoping — the replacement for the
old substring-prefix matching. See `Example/A2c.idr` for the worked pattern.

## Discovering the API

Use `:browse` to list a module's exports, and `:doc` / `:t` on any name:

In [ ]:
:browse Nn.Linear

Next: [03 Data and Loss](03_data_and_loss.ipynb) — typed training data and loss functions.